# Deepfake Detection

### A clear introduction to modern methods for detecting manipulated images and videos

Deepfake detection is a **binary classification problem** in which a model decides whether visual media is **real** or **synthetically manipulated**.

This notebook explains the main ideas behind deepfake detection, common model architectures, evaluation methods, and a practical PyTorch pipeline.


## 1. What is a Deepfake?

A **deepfake** is media that has been generated or manipulated using deep learning.

Common manipulations include:

- **Face swapping** — replacing one person's face with another.
- **Face reenactment** — changing facial expressions or mouth movement.
- **Lip synchronization** — modifying mouth motion to match new speech.
- **Fully synthetic faces** — generating a face that does not belong to a real person.

The detector does not usually try to reconstruct the original image. Instead, it learns visual or temporal patterns that distinguish **real** media from **fake** media.


## 2. Why Deepfake Detection Matters

Deepfake technology has useful creative applications, but it can also be misused.

Deepfake detection supports:

- digital media forensics
- misinformation analysis
- identity protection
- fraud detection
- content authenticity verification
- social-media safety

The technical challenge is that generation models continue to improve, so detectors must recognize subtle manipulation traces rather than only obvious visual defects.


## 3. How Deepfakes Are Generated

Modern deepfakes can be created with several families of generative models:

| Method | Main Idea |
|---|---|
| Autoencoders | Learn a compressed representation and reconstruct a target face |
| GANs | A generator creates fake samples while a discriminator tries to distinguish them |
| Diffusion Models | Generate or edit images through iterative denoising |
| Neural Rendering / Reenactment | Modify pose, expression, identity, or mouth motion |

A detector can exploit traces left by these generation and blending processes.


## 4. General Deepfake Detection Pipeline

A typical image-based detector follows this workflow:

```text
Input Image / Video
        ↓
   Face Detection
        ↓
Face Crop + Alignment
        ↓
   Preprocessing
        ↓
 Feature Extraction
        ↓
 Classification Model
        ↓
   Real  /  Fake
```

For video, the system may classify individual frames or analyze **temporal information across multiple frames**.


## 5. What Does a Detector Look For?

Deepfake detectors may learn several kinds of evidence.

### Spatial artifacts
- unnatural skin texture
- inconsistent facial boundaries
- blending artifacts
- unusual reflections
- local color inconsistencies
- imperfect eyes, teeth, or hair regions

### Frequency artifacts
Synthetic images can contain unusual patterns in the frequency domain that are difficult to notice visually.

### Temporal artifacts
Video deepfakes may contain inconsistent:
- blinking
- facial motion
- head pose
- lip movement
- frame-to-frame appearance

A strong detector should learn general manipulation cues rather than memorizing one dataset.


# 6. CNN-Based Detection

Convolutional Neural Networks (CNNs) are a natural starting point for deepfake detection because they learn local spatial patterns.

```text
Face Image
   ↓
Convolution
   ↓
ReLU
   ↓
Pooling
   ↓
Convolution
   ↓
ReLU
   ↓
Global Average Pooling
   ↓
Fully Connected Layer
   ↓
Real / Fake
```

Earlier CNN layers learn simple patterns such as edges and textures.  
Deeper layers learn more complex facial and manipulation features.


In [ ]:
import torch
import torch.nn as nn

class SimpleDeepfakeCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Linear(128, 2)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model = SimpleDeepfakeCNN()
model


## 7. XceptionNet

**XceptionNet** is an important architecture in deepfake detection research.

Instead of using standard convolutions only, Xception uses **depthwise separable convolutions**.

### Standard convolution
A standard convolution learns spatial and channel relationships together.

### Depthwise separable convolution
It separates the operation into:

```text
Input
  ↓
Depthwise Convolution
  ↓
Pointwise 1 × 1 Convolution
  ↓
Output
```

This reduces computation while still allowing the network to learn detailed visual features.

Xception became a common baseline for detecting manipulated facial images because it can capture fine-grained texture and blending artifacts.


## 8. EfficientNet

EfficientNet improves CNN scaling by balancing three dimensions:

- network **depth**
- network **width**
- input **resolution**

Instead of increasing only one dimension, EfficientNet scales them together.

```text
Input Face
    ↓
Stem Convolution
    ↓
MBConv Blocks
    ↓
Feature Extraction
    ↓
Global Average Pooling
    ↓
Classifier
    ↓
Real / Fake
```

EfficientNet-based detectors are useful when we want strong feature extraction without using an unnecessarily large model.


In [ ]:
from torchvision import models
import torch.nn as nn

# Pretrained EfficientNet backbone
efficientnet = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

# Replace the ImageNet classifier with a binary classifier
in_features = efficientnet.classifier[1].in_features
efficientnet.classifier[1] = nn.Linear(in_features, 2)

efficientnet


# 9. Vision Transformers for Deepfake Detection

CNNs focus strongly on local patterns.  
Vision Transformers can model relationships between distant parts of an image.

A Vision Transformer divides an image into patches:

```text
Image
  ↓
Image Patches
  ↓
Patch Embeddings
  ↓
Positional Embeddings
  ↓
Transformer Encoder
  ↓
Classification Token
  ↓
Real / Fake
```

The **self-attention** mechanism allows the model to compare information across different facial regions.

This can be useful when manipulation evidence is distributed across the whole face rather than concentrated in one small area.


# 10. Frequency-Based Detection

Some manipulation traces may be weak in normal RGB space but more visible after transforming the image into the frequency domain.

Two common transforms are:

- **FFT** — Fast Fourier Transform
- **DCT** — Discrete Cosine Transform

```text
RGB Image
   ↓
FFT / DCT
   ↓
Frequency Representation
   ↓
Feature Extractor
   ↓
Classifier
   ↓
Real / Fake
```

Frequency-based approaches can complement spatial detectors because generative pipelines may introduce periodic or spectral artifacts.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show_frequency_spectrum(image):
    """
    image: 2D grayscale NumPy array
    """
    spectrum = np.fft.fft2(image)
    spectrum = np.fft.fftshift(spectrum)
    magnitude = np.log1p(np.abs(spectrum))

    plt.figure(figsize=(6, 5))
    plt.imshow(magnitude, cmap="gray")
    plt.title("Frequency Spectrum")
    plt.axis("off")
    plt.show()


# 11. Temporal Deepfake Detection

A video contains information that a single image does not.

Instead of classifying each frame independently, temporal models analyze a sequence:

```text
Frame 1 ─┐
Frame 2 ─┤
Frame 3 ─┼→ CNN Feature Extractor → LSTM / Transformer → Real / Fake
Frame 4 ─┤
Frame N ─┘
```

Possible temporal clues include:

- inconsistent facial motion
- unnatural blinking
- unstable facial boundaries
- lip-sync errors
- sudden texture changes between frames

Common temporal models include **LSTM**, **GRU**, and **video transformers**.


# 12. Multimodal Deepfake Detection

Some deepfakes manipulate both video and audio.

A multimodal system can compare multiple information sources:

```text
Video Frames → Visual Encoder ─┐
                              ├→ Feature Fusion → Classifier
Audio Signal → Audio Encoder ─┘
```

The model can detect contradictions such as:

- mouth motion that does not match speech
- voice characteristics inconsistent with the visible speaker
- timing differences between audio and facial motion


# 13. Common Deepfake Datasets

| Dataset | Main Characteristics |
|---|---|
| FaceForensics++ | Multiple facial manipulation methods and compression settings |
| Celeb-DF | Higher-quality celebrity deepfake videos |
| DFDC | Large-scale dataset released for the Deepfake Detection Challenge |
| WildDeepfake | Deepfake content collected from real-world internet sources |

Dataset choice matters because a detector may perform well on familiar manipulations but fail on unseen ones.


# 14. Preprocessing

A detector usually does not feed an entire video frame directly into the network.

A common preprocessing pipeline is:

```text
Video
  ↓
Sample Frames
  ↓
Detect Face
  ↓
Crop Face
  ↓
Align Face
  ↓
Resize
  ↓
Normalize
  ↓
Model
```

Typical image preprocessing may include:

- resizing
- normalization
- horizontal flipping
- random cropping
- mild color augmentation

Aggressive augmentation should be used carefully because it can remove subtle forensic evidence.


In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# 15. Dataset Structure in PyTorch

For a simple image experiment, the dataset can be organized as:

```text
dataset/
│
├── train/
│   ├── real/
│   └── fake/
│
├── val/
│   ├── real/
│   └── fake/
│
└── test/
    ├── real/
    └── fake/
```

`ImageFolder` automatically assigns a class label to each folder.


In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Example:
# train_dataset = ImageFolder("dataset/train", transform=train_transform)
# val_dataset   = ImageFolder("dataset/val", transform=test_transform)
# test_dataset  = ImageFolder("dataset/test", transform=test_transform)

# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
# test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


# 16. Training

For binary classification with two output logits, we can use **CrossEntropyLoss**.

The training loop contains four main steps:

1. Forward pass
2. Compute loss
3. Backpropagation
4. Update model parameters


In [ ]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = SimpleDeepfakeCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_one_epoch(model, loader):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


# 17. Evaluation Metrics

Accuracy alone may not be enough.

For deepfake detection, useful metrics include:

### Accuracy
Overall fraction of correct predictions.

### Precision
Among samples predicted as fake, how many were actually fake?

### Recall
Among actual fake samples, how many were detected?

### F1-score
Harmonic mean of precision and recall.

### ROC-AUC
Measures how well the model separates real and fake samples across classification thresholds.

A confusion matrix also helps us understand the types of errors the model makes.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def calculate_metrics(y_true, y_pred, y_score):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
        "confusion_matrix": confusion_matrix(y_true, y_pred)
    }


# 18. The Generalization Problem

One of the most important problems in deepfake detection is **generalization**.

A detector can achieve strong results on the same type of data used during training but perform poorly when tested on:

- another dataset
- another manipulation technique
- stronger compression
- different resolutions
- new generation models

This happens when the detector learns **dataset-specific shortcuts** instead of general manipulation evidence.

A stronger evaluation therefore includes **cross-dataset testing**.

Example:

```text
Train: FaceForensics++
            ↓
        Detector
            ↓
Test: Celeb-DF / WildDeepfake
```

Cross-dataset evaluation is often more informative than testing only on a random split of the same dataset.


# 19. Explainability

Deepfake detectors should not only produce a prediction; it can also be useful to inspect **where the model is looking**.

Methods such as **Grad-CAM** highlight image regions that contribute strongly to a CNN prediction.

```text
Input Face
    ↓
CNN Prediction
    ↓
Grad-CAM
    ↓
Attention Heatmap
```

This can help researchers determine whether a model is focusing on meaningful facial manipulation regions or irrelevant background shortcuts.


# 20. Comparison of Detection Approaches

| Approach | Main Strength | Main Limitation |
|---|---|---|
| CNN | Strong local texture modeling | May learn dataset-specific artifacts |
| Xception | Effective fine-grained feature extraction | Cross-dataset robustness can still be limited |
| EfficientNet | Good accuracy-efficiency balance | Still mainly spatial |
| Vision Transformer | Captures global relationships | Usually needs more data and computation |
| Frequency-based | Detects hidden spectral artifacts | Can be sensitive to compression |
| Temporal | Uses motion across frames | More computationally expensive |
| Multimodal | Combines complementary evidence | Requires synchronized modalities |


# 21. Key Challenges

Current deepfake detection research faces several important challenges:

1. **Unseen manipulations**  
   New generation methods may produce artifacts that were absent from the training data.

2. **Compression**  
   Social-media compression can destroy weak forensic traces.

3. **Domain shift**  
   Lighting, camera quality, resolution, and demographics can differ between datasets.

4. **Adversarial adaptation**  
   Generators can be improved specifically to remove known detection cues.

5. **Real-world reliability**  
   Laboratory accuracy does not automatically translate to robust deployment.


# 22. Summary

Deepfake detection is a rapidly evolving computer vision and multimedia forensics problem.

The main ideas covered in this notebook are:

- deepfakes can be detected using spatial, frequency, temporal, or multimodal evidence
- CNNs remain useful baselines for image-based detection
- Xception and EfficientNet are common feature extractors
- transformers can model global image relationships
- temporal models use information across video frames
- cross-dataset generalization is one of the most important evaluation challenges
- explainability can help inspect what a detector has actually learned

The goal of modern deepfake detection is not only high accuracy on familiar data, but also **robust detection of unseen manipulations in realistic conditions**.


# References

1. Rössler, A. et al. (2019). *FaceForensics++: Learning to Detect Manipulated Facial Images*. ICCV.
2. Chollet, F. (2017). *Xception: Deep Learning with Depthwise Separable Convolutions*. CVPR.
3. Tan, M. & Le, Q. (2019). *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks*. ICML.
4. Li, Y. et al. (2020). *Celeb-DF: A Large-scale Challenging Dataset for DeepFake Forensics*. CVPR.
5. Dolhansky, B. et al. (2020). *The Deepfake Detection Challenge (DFDC) Dataset*.
6. Dosovitskiy, A. et al. (2021). *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*. ICLR.
